# Notebook: Introduction to Prompting Strategies for Large Language Models (LLMs)

This notebook introduces various prompting strategies that can help improve the performance of Large Language Models (LLMs).

## 📚 Sources

- [Prompting Guide](https://www.promptingguide.ai/techniques)
- [Python Package `openai`](https://github.com/openai/openai-python)

---

Have fun exploring and experimenting! 🤗

We use the `openai` package to interact with an LLM. This package allows communication with both the OpenAI API and other LLMs that offer a compatible API. 

**Important Note:** At the university, we have provided several Large Language Models via a server. I refer to the VPN access instructions that I have uploaded to the learning management system. Requests to the LLM are only possible **within the university network**, which means you are either on campus in `eduroam` or connected via VPN.

The university server offers several open-source LLMs, which are provided via Ollama.

We will now install the package.

In [1]:
%%capture
!pip install openai

In [2]:
# Now let's import the package
from openai import OpenAI

In [ ]:
# Let's define the API URL and API key. Note the hint above!
LLM_URL = "http://localhost:11434/v1"
LLM_MODEL = "gemma3:4b"

In [4]:
client = OpenAI(
    base_url=LLM_URL,
    api_key='ollama',
)

In [5]:
# Let's define a function that makes a request to the LLM. This is the completions API, which performs next-token prediction.
# A maximum of `max_tokens` are generated, which can be limited by the `stop` token.
def llm_completion(prompt, temperature=0.0, stop=["\n"], seed=0, max_tokens=256):
    completion = client.completions.create(
        model=LLM_MODEL, # The LLM model being used
        temperature=temperature, # Temperature controls the creativity of responses, more on this next week
        stop=stop, # As soon as one of the stop tokens is generated, the LLM stops generating
        prompt=prompt, # The input text that the LLM should respond to
        max_tokens=max_tokens, # Maximum number of tokens to generate
        seed=seed # Random seed for reproducibility
    )
    return completion.choices[0].text

In [6]:
# Let's define another function that uses the Chat Completions API. This allows passing messages in a chat format.
def llm_completion_chat(prompt, temperature=0.0, stop=None, image_url=None, seed=0):
    messages = [
        {
            "role": "user",
            "content": [
                    {
                        "type": "text",
                        "text": prompt
                    }
            ] + ([{"type": "image_url", "image_url": {"url": image_url}}] if image_url else [])
        }
    ]

    extra_body = {"stop": [stop]} if stop else {}

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=messages,
        temperature=temperature,
        stream=False,
        seed=seed,
        extra_body=extra_body,
    )

    return response.choices[0].message.content

## 1. Zero-shot Prompting

**Zero-shot Prompting** means you directly ask the LLM to perform a task – without any examples or additional explanations.  

The model uses its trained knowledge to answer your query as best as possible. This often works surprisingly well, especially with clearly formulated questions.

In [7]:
prompt = "Paris is the capital of?"
print(llm_completion_chat(prompt, stop="\n"))

Paris is the capital of **France**. 


## 2. Few-shot Prompting

**Few-shot Prompting** means you provide the LLM with a few examples to give more context.

This helps the model better understand your query and generate more relevant answers.

In this context, the paper **"Language Models are Few-Shot Learners"** by Brown et al. (2020), which introduced GPT-3 in 2020, is of particular significance. Instead of retraining the model, it is sufficient for LLMs to provide a few examples directly in the prompt (Few-Shot) to solve new tasks. This can be described as a paradigm shift: Instead of fine-tuning a model for each task, large models can generalize through prompt engineering.

[Link to the paper](https://proceedings.neurips.cc/paper_files/paper/2020/file/1457c0d6bfcb4967418bfb8ac142f64a-Paper.pdf)

In [8]:
# In the following example, a prompt is used to extract sentiment elements from sentences.
# The model should identify the relevant elements and classify their sentiment.
prompt = '''Sentence: It was really great in Berlin.
Sentiment Elements: [("Berlin", "positive")]
Sentence: The food wasn't very tasty.
Sentiment Elements: [("food", "negative")]
Sentence: Here in Portugal there are fantastic beaches but unfortunately the weather was bad.
Sentiment Elements: [("beaches", "positive"), ("weather", "negative")]
Sentence: The city is very beautiful.
Sentiment Elements: '''

# stop is an optional stop criterion to end the response
print(llm_completion(prompt, stop="\n"))  # Stops the response at "\n"

[("city", "positive")]


## 3. Chain of Thought

Introduced in [Wei et al. (2022)](https://arxiv.org/abs/2201.11903), Chain-of-Thought (CoT) prompting enables complex reasoning capabilities through intermediate steps in the thinking process. You can combine it with few-shot prompting to achieve better results on more complex tasks that require reasoning before answering.

In [9]:
prompt = '''I went to the market and bought 10 apples. I gave 2 apples to the neighbor and 2 to the repairman. Then I went and bought 5 more apples and ate 1. How many apples did I have left?
Let's think step by step.\n'''

print(llm_completion_chat(prompt))

Okay, let's break this down step by step:

1. **Started with:** You began with 10 apples.
2. **Gave to neighbor:** You gave away 2 apples (10 - 2 = 8).
3. **Gave to repairman:** You gave away another 2 apples (8 - 2 = 6).
4. **Bought more:** You bought 5 more apples (6 + 5 = 11).
5. **Ate one:** You ate 1 apple (11 - 1 = 10).

**Answer:** You have 10 apples left.


## 4. Self-Consistency / Majority Vote

Self-Consistency is a technique introduced in [Wang et al. (2022)](https://arxiv.org/abs/2203.11171). It leverages the fact that LLMs can often generate multiple plausible answers when responding to questions. Instead of relying on a single answer, Self-Consistency aggregates multiple answers and selects the most frequent answer to increase accuracy.

In [10]:
import re

prompt = '''Q: Anna has 2 apples. She gets 3 more apples from her friend. How many apples does she have now?
A: Anna has 2 apples. She gets 3 more. 2 + 3 = 5. The answer is 5.

Q: Tom has 7 colored pencils. He buys 5 more. How many colored pencils does he have in total?
A: Tom has 7 colored pencils. He buys 5 more. 7 + 5 = 12. The answer is 12.

Q: Lisa has 10 euros. She gets 4 euros pocket money. How much money does she have afterwards?
A: Lisa has 10 euros. She gets 4 euros more. 10 + 4 = 14. The answer is 14.

Q: Paul has 3 chocolate bars. He gets 6 chocolate bars from his brother. How much chocolate does Paul have?
A: '''

# We can now invoke Self-Consistency
predictions = [llm_completion(prompt, stop="\n\n", seed=i, temperature=0.8) for i in range(5)]

prediction_ints = []
for p in predictions:
    numbers = re.findall(r'\d+', p)
    if numbers:
        # We take the last found number as the prediction
        prediction_ints.append(int(numbers[-1]))
    else:
        prediction_ints.append(None)

# Output: five predictions + most frequent answer
prediction_ints, max(set(prediction_ints), key=prediction_ints.count)

([9, 9, 9, 9, 9], 9)

## 5. Generated Knowledge Prompting

Large language models (LLMs) are continuously being improved, and a popular technique involves the ability to incorporate knowledge or information to help the model make more accurate predictions.

Can the model also be used with a similar idea to generate knowledge before making a prediction? This is exactly what is attempted in the paper by [Liu et al. 2022](https://arxiv.org/pdf/2110.08387.pdf) – generating knowledge to be used as part of the prompt. How useful is this for tasks like common sense reasoning?

Steps:

1. First, we generate some "knowledge statements"
2. Then we use these knowledge statements to extend the prompt
3. Finally, we use the extended prompt to make the prediction

In [11]:
prompt = '''Input: Greece is larger than Mexico.
Knowledge: Greece is approximately 131,957 square kilometers, while Mexico is approximately 1,964,375 square kilometers. Mexico is therefore 1,389% larger than Greece.

Input: Glasses always fog up.
Knowledge: Condensation occurs on eyeglass lenses when water vapor from your sweat, breath, and ambient moisture hits a cold surface, cools, and then turns into tiny liquid droplets, forming a film that you perceive as fog. Your lenses will be relatively cool compared to your breath, especially when the outside air is cold.

Input: A fish is capable of thinking.
Knowledge: Fish are more intelligent than they appear. In many areas, such as memory, their cognitive abilities match those of 'higher' vertebrates, including non-human primates. The long-term memories of fish help them keep track of complex social relationships.

Input: A common effect of smoking many cigarettes over a lifetime is an above-average probability of getting lung cancer.
Knowledge: Those who consistently smoked less than one cigarette per day over their lifetime had a nine times higher risk of dying from lung cancer than non-smokers. For people who smoked between one and 10 cigarettes per day, the risk of dying from lung cancer was almost 12 times higher than for non-smokers.

Input: How many planets are there in the solar system?
Knowledge: '''

knowledge = llm_completion(prompt, stop="\n")

prompt_with_knowledge = f'''Question: Is Earth the only planet in the solar system?
Knowledge: {knowledge}
Answer:'''

answer = llm_completion(prompt_with_knowledge, stop="\n")
print(prompt_with_knowledge + answer)

Question: Is Earth the only planet in the solar system?
Knowledge: There are eight planets in our solar system: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptune.
Answer:No, Earth is not the only planet in the solar system. There are eight planets: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptune.


## Exercises

### Exercise 1: Few-Shot Prompting for Text Classification

Write a few-shot prompt to ask the LLM to classify the sentiment (positive, negative, neutral) of a given text. Use at least three examples in your prompt.

In [12]:
# Your code here...

<details>
<summary><b>Show Solution</b></summary>

```python
prompt = '''Sentence: It was really great in Berlin.
Label: positive
Sentence: The food wasn't very tasty.
Label: negative
Sentence: Here in Portugal there are fantastic beaches.
Label: positive
Sentence: Unfortunately the weather was bad.
Label: '''

print("Label:", llm_completion(prompt, stop="\n"))
```

</details>

### Exercise 2: Named Entity Recognition with Few-Shot Learning

#### Objective
Develop a few-shot prompt strategy for the automatic recognition of **location names** (LOC - Location Entities) in English texts.  
The prepared file **`LOC_sentences.txt`** serves as the data basis, which contains sentences in the format  

```
Sentence####["Entity1", "Entity2", ...]
```

Each entry consists of an English sentence and the associated list of all location names (LOC entities) in that sentence.  

#### Requirements
1. **Few-Shot Learning:** Create a prompt with **five random few-shot examples** from `LOC_sentences.txt` that demonstrate the desired input-output behavior.  
2. **Output Format:** The model should return recognized location names as a string list (e.g., `["Berlin", "Munich"]`).  
3. **Evaluation:** Test your few-shot prompting on 64 random examples from `LOC_sentences.txt`. 

#### Evaluation with Accuracy
To evaluate the results, **accuracy** should be calculated.  

A sentence is considered **correctly labeled** if the predicted list of location names exactly matches the gold standard list from `LOC_sentences.txt`.

#### Example of Expected Behavior

```md
Input: "In Essen the two brothers met."
Output: ["Essen"]
```

Tip: You can use `eval()` to convert the model's string output into a Python list.

#### Example Prompt

```text
Recognize all location names in English sentences and return them as a list.

Sentence: The renewed administrative reform of 1815 once again brought a new district assignment for Passenheim.
Location names: ['Passenheim']

Sentence: The main base was relocated from Alderney to Jersey in 2006.
Location names: ['Alderney', 'Jersey']

Sentence: It initially runs along Bayernstraße, crosses it, and reaches the Dutzendteich stop, where there is an interchange to the S-Bahn to Altdorf.
Location names: ['Bayernstraße', 'Dutzendteich', 'Altdorf']

Sentence: Rhein-Kreis Neuss Actually, the topic always comes up at the beginning of the year when the parliamentary groups discuss the budget: Should the Rhein-Kreis Neuss sell its RWE shares?
Location names: ['Rhein-Kreis Neuss', 'Rhein-Kreis Neuss']

Sentence: A Goevier is on display at the Flugwerft Schleißheim.
Location names: ['Schleißheim']

Sentence: Today, the course of the northwestern wall ring adjoining the tower is marked by a red brick stripe, while the southwestern wall ring is still intact up to the grounds of the former Franciscan monastery St. Johannis.
Location names:
```

In [13]:
# Load the dataset
dataset = []

with open("../content/LOC_sentences.txt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        sentence, entities_str = line.split("####")
        entities = eval(entities_str)
        dataset += [[sentence, entities]]

print("Sentence:", dataset[0][0])
print("Entities:", dataset[0][1])

# 5 random few-shot examples
import random
random.seed(42)
few_shot_examples = random.sample(dataset, 5)
test_data = [item for item in dataset if item not in few_shot_examples][:64]

Sentence: Today, the course of the northwestern wall ring adjoining the tower is marked by a red brick stripe, while the southwestern wall ring is still intact up to the grounds of the former Franciscan monastery St. Johannis.
Entities: ['St . Johannis']


In [14]:
# Your code here...

<details>
<summary><b>Show Solution</b></summary>

```python
# Create prompt
def create_prompt(examples, test_sentence):
    prompt = "Recognize all location names in English sentences and return them as a list.\n\n"
    
    for sentence, entities in examples:
        prompt += f"Sentence: {sentence}\nLocation names: {entities}\n\n"
    
    prompt += f"Sentence: {test_sentence}\nLocation names:"
    return prompt

# Evaluation
def evaluate(test_data):
    correct = 0
    total = len(test_data)
    
    for sentence, true_entities in test_data:
        prompt = create_prompt(few_shot_examples, sentence)
        response = llm_completion(prompt, stop="\n")
        
        # Since the LLM sometimes returns invalid Python expressions, we use try-except
        try:
            predicted_entities = eval(response.strip())
            if set(predicted_entities) == set(true_entities): # Use set to ignore order
                correct += 1
        except:
            pass
    
    accuracy = correct / total
    return accuracy

# Execution
accuracy = evaluate(test_data) * 100
print(f"Accuracy: {accuracy:.2f}")
```

</details>